# Lesson 02 - Microsoft Agent Framework ਦੀ ਖੋਜ

**Microsoft Agent Framework (MAF)** ਏਕਾਈਕ ਫ੍ਰੇਮਵਰਕ ਹੈ ਜੋ AI ਏਜੰਟਾਂ ਬਣਾਉਣ ਲਈ ਹੈ। ਇਹ ਚਾਰ ਮੁੱਖ ਭਾਗਾਂ ਨਾਲ ਇੱਕ ਸਾਫ਼, ਸੰਯੋਜਯੋਗ ਆਰਕੀਟੈਕਚਰ ਪ੍ਰਦਾਨ ਕਰਦਾ ਹੈ:

- **ਕਲਾਇੰਟ** – ਏਕ AI ਮਾਡਲ ਏਂਡਪੌਇੰਟ ਨਾਲ ਜੁੜਦਾ ਹੈ ਅਤੇ ਸੰਚਾਰ ਸੰਭਾਲਦਾ ਹੈ
- **ਏਜੰਟ** – ਕਲਾਇੰਟ ਨੂੰ ਹੁਕਮਾਂ ਅਤੇ ਟੂਲ ਪਰਿਭਾਸ਼ਾਵਾਂ ਨਾਲ ਘੇਰਦਾ ਹੈ
- **ਟੂਲਜ਼** – ਮਾਡਲ ਦੁਆਰਾ ਕਾਲ ਕੀਤੇ ਜਾ ਸਕਣ ਵਾਲੇ ਕਸਟਮ ਫੰਕਸ਼ਨਾਂ ਨਾਲ ਏਜੰਟ ਦੀਆਂ ਸਮਰੱਥਾਵਾਂ ਨੂੰ ਵਧਾਉਂਦੇ ਹਨ
- **ਸੈਸ਼ਨ** – ਬਹੁ-ਮੁੜ-ਵਾਰਤਾਲਾਪ ਲਈ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਬਰਕਰਾਰ ਰੱਖਦਾ ਹੈ

ਇਸ ਪਾਠ ਵਿੱਚ, ਅਸੀਂ ਇੱਕ **ਟਰੈਵਲ ਬੁਕਿੰਗ ਏਜੰਟ** ਬਣਾਵਾਂਗੇ ਜੋ ਇਨ੍ਹਾਂ ਧਾਰਣਾਵਾਂ ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਗੰਤੀਸ਼ੀਲਤਾ ਦੀ ਜਾਂਚ ਕਰਦਾ ਹੈ।


## ਸੈਟਅਪ


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## ਏਜੰਟ ਫਰੇਮਵਰਕ ਆਰਕੀਟੈਕਚਰ ਨੂੰ ਸਮਝਣਾ

ਮਾਈਕਰੋਸਾਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਇੱਕ ਪਰਤਦਾਰ ਆਰਕੀਟੈਕਚਰ ਨਾਲ ਪਾਲਣਾ ਕਰਦਾ ਹੈ:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **ਕਲਾਇੰਟ** – ਇੱਕ `AzureAIProjectAgentProvider` ਇਕ Azure OpenAI ਡਿਪਲੋਇਮੈਂਟ ਨਾਲ ਜੁੜਦਾ ਹੈ। ਇਹ ਪ੍ਰਮਾਣੀਕਰਨ, ਬੇਨਤੀ ਫਾਰਮੈਟਿੰਗ ਅਤੇ ਜਵਾਬ ਪਾਰਸਿੰਗ ਨੂੰ ਸੰبھਾਲਦਾ ਹੈ।
2. **ਏਜੰਟ** – ਕਲਾਇੰਟ ਤੋਂ `provider.create_agent()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ, ਏਜੰਟ ਮਾਡਲ ਪਹੁੰਚ ਨੂੰ ਹਦਾਇਤਾਂ (ਸਿਸਟਮ ਪ੍ਰੌਂਪਟ) ਅਤੇ ਟੂਲਜ਼ ਨਾਲ ਜੋੜਦਾ ਹੈ।
3. **ਟੂਲਜ਼** – ਪਾਈਥਨ ਫੰਕਸ਼ਨ ਜੋ `@tool` ਨਾਲ ਸਜਾਏ ਗਏ ਹਨ ਜਿਨ੍ਹਾਂ ਨੂੰ ਏਜੰਟ ਕਾਰਵਾਈ ਕਰਨ ਜਾਂ ਡਾਟਾ ਪ੍ਰਾਪਤ ਕਰਨ ਲਈ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।
4. **ਸੈਸ਼ਨ** – ਇੱਕ `AgentSession` ਆਬਜੈਕਟ (ਜੋ `agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਜੋ ਗੱਲਬਾਤ ਦਾ ਇਤਿਹਾਸ ਸੰਜੋਇਆ ਰੱਖਦਾ ਹੈ, ਇਸ ਨਾਲ ਏਜੰਟ ਪਿਛਲੇ ਸੰਦਰਭ ਨੂੰ ਯਾਦ ਰੱਖਦਾ ਹੈ ਅਤੇ ਬਹੁ-ਚਰਣ ਵਾਲੀ ਸੰਵਾਦਗਤਕਤਾ ਨੂੰ ਯੋਗ ਬਨਾਉਂਦਾ ਹੈ।

ਆਓ, ਹਰ ਪਰਤ ਨੂੰ ਕਦਮ ਦਰ ਕਦਮ ਬਣਾਈਏ।


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool ਡੈਕੋਰੇਟਰ ਨਾਲ ਟੂਲ ਸ਼ਾਮਿਲ ਕਰਨਾ

ਟੂਲ ਏਜੰਟਾਂ ਨੂੰ ਲਿਖਤ ਬਣਾਉਣ ਤੋਂ ਇਲਾਵਾ ਕਿਰਿਆਵਾਂ ਕਰਨ ਦੇ ਯੋਗ ਬਣਾਉਂਦੇ ਹਨ। `@tool` ਡੈਕੋਰੇਟਰ ਇੱਕ ਆਮ ਪਾਇਥਨ ਫੰਕਸ਼ਨ ਨੂੰ ਕੁਝ ਅਜਿਹਾ ਬਣਾ ਦਿੰਦਾ ਹੈ ਜਿਸਨੂੰ ਏਜੰਟ ਕਾਲ ਕਰ ਸਕਦਾ ਹੈ।

ਮੁੱਖ ਬਿੰਦੂ:
- ਮਾਡਲ ਹਰ ਪੈਰਾਮੀਟਰ ਨੂੰ ਸਮਝਣ ਲਈ `Annotated[type, "description"]` ਵਰਤੋਂ।
- ਡੌਕਸਟ੍ਰਿੰਗ ਟੂਲ ਦਾ ਵਰਣਨ ਬਣ ਜਾਂਦੀ ਹੈ ਜੋ ਮਾਡਲ ਵੇਖਦਾ ਹੈ।
- `approval_mode="never_require"` ਦਾ ਮਤਲਬ ਹੈ ਕਿ ਟੂਲ ਬਿਨਾਂ ਯੂਜ਼ਰ ਦੀ ਪੁਸ਼ਟੀ ਦੇ ਖੁਦਮੁਖਤਿਆਰ ਚੱਲਦਾ ਹੈ।


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ਟੂਲਸ ਨਾਲ ਏਜੰਟ ਬਣਾਉਣਾ

ਹੁਣ ਅਸੀਂ ਕਲਾਇੰਟ, ਹدایਤਾਂ ਅਤੇ ਟੂਲਸ ਨੂੰ ਇੱਕ ਏਜੰਟ ਵਿੱਚ ਮਿਲਾ ਦਿੰਦੇ ਹਾਂ। `instructions` ਸਿਸਟਮ ਪ੍ਰਾਂਪਟ ਵਜੋਂ ਕੰਮ ਕਰਦੇ ਹਨ — ਇਹ ਏਜੰਟ ਦੇ ਵਿਅਕਤੀਗਤ ਅਤੇ ਵਿਹਾਰ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦੇ ਹਨ।


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## ਸੈਸ਼ਨ ਨਾਲ ਬਹੁ-ਮੋੜੀ ਗੱਲਬਾਤ

ਇੱਕ `AgentSession` (`agent.create_session()` ਰਾਹੀਂ ਬਣਾਇਆ ਗਿਆ) ਗੱਲਬਾਤ ਵਿੱਚ ਸਾਰੇ ਸੁਨੇਹਿਆਂ ਦਾ ਟ੍ਰੈਕ ਰੱਖਦਾ ਹੈ। ਹਰ `agent.run()` ਕਾਲ ਨੂੰ ਉਹੀ ਸੈਸ਼ਨ ਦੇ ਕੇ, ਏਜੰਟ ਕੋਲ ਪੂਰੇ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਦੀ ਪਹੁੰਚ ਹੁੰਦੀ ਹੈ ਅਤੇ ਉਹ ਪਹਿਲਾਂ ਦੇ ਸੁਨੇਹਿਆਂ ਨੂੰ ਵੇਖ ਸਕਦਾ ਹੈ।

ਅਸੀਂ `tools=[check_destination_availability]` ਪਾਸ ਕਰਦੇ ਹਾਂ ਤਾਂ ਜੋ ਏਜੰਟ ਹਰ ਮੋੜ 'ਤੇ ਸਾਡਾ ਉਪਲਬਧਤਾ ਚੈੱਕਰ ਕਾਲ ਕਰ ਸਕੇ।


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## ਸੰਖੇਪ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਮਾਈਕਰੋਸੌਫਟ ਏਜੰਟ ਫਰੇਮਵਰਕ ਦੇ ਚਾਰ ਸਤੰਭਾਂ ਦੀ ਖੋਜ ਕੀਤੀ:

| ਸੰਕਲਪ | ਤੁਸੀਂ ਕੀ ਸਿੱਖਿਆ |
|---------|------------------|
| **ਕਲਾਇੰਟ** | `AzureAIProjectAgentProvider` ਪ੍ਰਮਾਣ ਪੱਤਰ-ਆਧਾਰਿਤ ਪ੍ਰਮਾਣਿਕਤਾ ਨਾਲ Azure OpenAI ਨਾਲ ਜੁੜਦਾ ਹੈ |
| **ਏਜੰਟ** | `provider.create_agent()` ਇੱਕ ਮਾਡਲ ਕਨੈਕਸ਼ਨ ਨੂੰ ਹਦਾਇਤਾਂ ਅਤੇ ਨਾਮ ਦੇ ਨਾਲ ਬੰਡਲ ਕਰਦਾ ਹੈ |
| **ਉਪਕਰਣ** | `@tool` ਡੈਕੋਰੇਟਰ ਪਾਇਥਨ ਫੰਕਸ਼ਨਜ਼ ਨੂੰ ਏਜੰਟ ਦੇ ਕਾਲ ਕਰਨ ਲਈ ਉਪਲਬਧ ਕਰਵਾਉਂਦਾ ਹੈ |
| **ਸੈਸ਼ਨ** | `agent.create_session()` ਕਈ ਵਾਰੀ ਗੱਲਬਾਤ ਇਤਿਹਾਸ ਨੂੰ ਬਣਾਈ ਰੱਖਦਾ ਹੈ |

ਇਹ ਨਿਰਮਾਣ ਬਲਾਕ ਮਿਲ ਕੇ ਐਜੰਟਾਂ ਬਣਾਉਂਦੇ ਹਨ ਜੋ ਕੁਦਰਤੀ ਗੱਲਬਾਤ ਕਰ ਸਕਦੇ ਹਨ, ਬਾਹਰੀ ਫੰਕਸ਼ਨਾਂ ਨੂੰ ਕਾਲ ਕਰਦੇ ਹਨ, ਅਤੇ ਸੰਦਰਭ ਨੂੰ ਬਣਾਈ ਰੱਖਦੇ ਹਨ — ਜਿਹੜਾ ਬਾਅਦ ਦੇ ਪਾਠਾਂ ਵਿੱਚ ਹੋਰ ਅੱਗੇ ਦੇ ਐਜੰਟਿਕ ਨਮੂਨੇ ਲਈ ਆਧਾਰ ਹੈ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰਿਆ ਗਿਆ ਬਿਆਨ**:  
ਇਹ ਦਸਤਾਵੇਜ਼ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਅਨੁਵਾਦ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾ ਲਈ ਕੋਸ਼ਿਸ਼ ਕਰਦੇ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਸੂਚਿਤ ਰਹੋ ਕਿ ਆਟੋਮੈਟਿਡ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਕਾਰਗਤਾ ਹੋ ਸਕਦੀ ਹੈ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਹੀ ਅਥਾਰਟੀਟਾਵ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਮਹੱਤਵਪੂਰਣ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਿਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੀ ਵਰਤੋਂ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀ ਕਿਸੇ ਵੀ ਗਲਤਫਹਮੀ ਜਾਂ ਗਲਤ ਸਮਝ ਲਈ ਜ਼ਿੰਮੇਵਾਰ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
